# Create Arthritis National Research Foundation (ANRF) Awards

Creates Arthritis National Research Foundation (US, Irvine CA) awards from ANRF's own **master list of
every grant it has made, 1973-2026**, sent to OpenAlex by Laura Douglas (Senior Manager of Grants and
Impact) on 2026-10-26: **285 grants to 273 scholars, USD 35,959,951.57**. ANRF publishes no grants
export; its website (curearthritis.org scientist database, 203 cards for 2013-14..2026-27, plus 259
scientist spotlight pages, many 1990s-2012) lists a subset of the same grants and no grant missing from
the list. `scripts/local/anrf_to_s3.py` ships the merged file (list authoritative; website adds bio URL
as `landing_page_url` for 265, research or lay summary as `description` for 151, named fellowship for 27,
and seven documented corrections in `build_awards.py` OVERRIDES: row 143 title (sheet copied row 142's),
row 93 given-name typo, and rows 220-223/225 whose sheet "title" is a one-sentence summary, replaced by
ANRF's own year-2 database title for the same grant).

**Programmes (`funder_scheme`, from the list's Grant Code):** ALL = Arthritis and Related Autoimmune
Disease Research Grant (266); PSA = Psoriatic Arthritis Research Grant (6, industry-supported:
Janssen / J&J / BMS); AFAR = Arthritis and Aging Research Grant with the American Federation for
Aging Research (5); PDF = Postdoctoral Research Fellowship (4, `funding_type` fellowship); NPF =
Psoriatic Arthritis Collaborative Grant with the National Psoriasis Foundation (2); PEDRA = Rheum for
Kids: Pediatric Skin and Joint Grant with PeDRA (2). Co-funded grants stay under ANRF with the partner
named in `funder_scheme`.

**Amounts:** the list's per-grant total in USD (multi-year totals written as formulas such as
=75000+100000 were evaluated in the build). 100% coverage.

**Dates:** the list gives grant years only ("2013-15" = grant years 2013-14 and 2014-15), so
`start_year`/`end_year` are set and `start_date`/`end_date` are NULL (how-to §12: don't invent a day).

**`funder_award_id`:** ANRF publishes no grant numbers; synthetic
`ANRF-{start year}-{grant code}-{LEAD FAMILY}-{given initial}` (e.g. ANRF-2026-ALL-WOBMA-H). Checked
2026-10-26: 285 distinct ids, 0 collisions with `openalex_awards_raw` / `openalex_awards`, so 0 of the
647 existing F4320306227 citation stubs collapse (those are Crossref shells, ~70% India's Anusandhan
NRF; oxjob #1846).

**Prerequisites:** run `scripts/local/anrf_to_s3.py` first. It uploads
`s3://openalex-ingest/awards/anrf/anrf_projects.parquet` (§1.4 shrink guard).

**Funder details (Path A, F4320* Crossref-registered):**
- funder_id: `4320306227`
- display_name: Arthritis National Research Foundation
- ror_id / doi: from `openalex.funders.funders` (ror 03abbex33, doi 10.13039/100000964)

**Priority:** `601` (direct funder ingest; higher wins under the 2026-06-20 DESC dedup).

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.anrf_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/anrf/anrf_projects.parquet`;

In [ ]:
%sql
SELECT grant_code, funder_scheme, excluded, COUNT(*) as n, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) as usd
FROM openalex.awards.anrf_raw GROUP BY 1, 2, 3 ORDER BY n DESC;

In [ ]:
%sql
DESCRIBE openalex.awards.anrf_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.anrf_raw LIMIT 5;

## Step 1.6: Funder existence check (Path A)
F4320306227 is a Crossref-registered funder, so it MUST resolve to exactly 1
row in `openalex.funders.funders`. If 0 rows, STOP (do not proceed) and flag.

In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi, country_code
FROM openalex.funders.funders
WHERE funder_id = 4320306227;

## Step 2: Create ANRF Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.anrf_awards
USING delta
AS
WITH
src_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.funders.funders
    WHERE funder_id = 4320306227  -- Arthritis National Research Foundation
)

SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000 as id,
    NULLIF(TRIM(g.title), '') as display_name,
    NULLIF(TRIM(g.description), '') as description,
    f.funder_id,
    TRIM(g.funder_award_id) as funder_award_id,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN TRY_CAST(g.amount AS DOUBLE) END as amount,
    CASE WHEN TRY_CAST(g.amount AS DOUBLE) > 0 THEN 'USD' END as currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) as id,
        f.display_name,
        f.ror_id,
        f.doi
    ) as funder,
    COALESCE(NULLIF(TRIM(g.funding_type), ''), 'research') as funding_type,
    NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,
    'anrf_grants_master_list' as provenance,
    CAST(NULL AS DATE) as start_date,   -- grant years only (§12)
    CAST(NULL AS DATE) as end_date,
    TRY_CAST(g.start_year AS INT) as start_year,
    TRY_CAST(g.end_year AS INT) as end_year,
    -- struct field order MUST match openalex_awards_raw.
    named_struct(
        'given_name', NULLIF(TRIM(g.lead_given_name), ''),
        'family_name', NULLIF(TRIM(g.lead_family_name), ''),
        'orcid', CAST(NULL AS STRING),
        'role_start', CAST(NULL AS DATE),
        'affiliation', named_struct(
            'name', NULLIF(TRIM(g.institution), ''),
            'country', CAST(NULL AS STRING),
            'ids', CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>)
        )
    ) as lead_investigator,
    CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
    NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
    CAST(NULL AS STRING) as doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(TRIM(g.funder_award_id))))) % 9000000000) as works_api_url,
    current_timestamp() as created_date,
    current_timestamp() as updated_date
FROM openalex.awards.anrf_raw g
CROSS JOIN src_funder f
WHERE g.excluded = '0'
  AND g.funder_award_id IS NOT NULL AND TRIM(g.funder_award_id) != ''
  AND NULLIF(TRIM(g.title), '') IS NOT NULL;

In [ ]:
%sql
-- Shape check: every funder_award_id is the synthetic ANRF-<year>-<code>-<FAMILY>-<I>; expect 0 rows.
SELECT funder_award_id FROM openalex.awards.anrf_awards
WHERE NOT funder_award_id RLIKE '^ANRF-(19|20)[0-9]{2}-[A-Z]+-[A-Z]+-[A-Z](-[0-9]+)?$'
LIMIT 20;

In [ ]:
%sql
-- Must be 0: no id is shared with another provenance already in the raw table (hash or key collision).
SELECT a.funder_award_id, r.funder_id, r.funder_award_id AS other_key, r.provenance, r.priority
FROM openalex.awards.anrf_awards a
JOIN openalex.awards.openalex_awards_raw r
  ON r.id = a.id AND r.provenance != 'anrf_grants_master_list';

In [ ]:
%sql
-- Never shrink what may already be published: refuse to replace a larger earlier load.
SELECT assert_true(
    (SELECT COUNT(*) FROM openalex.awards.anrf_awards) >=
    (SELECT COUNT(*) FROM openalex.awards.openalex_awards_raw WHERE provenance = 'anrf_grants_master_list' AND priority = 601),
    'anrf_awards is smaller than the rows already in openalex_awards_raw; refusing to shrink'
) as no_shrink;

In [ ]:
%sql
-- Remove previous data for this source before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'anrf_grants_master_list' AND priority = 601;

-- Insert into openalex_awards_raw with priority.
-- Priority 601: direct-from-funder ingest of ANRF's master grants list.
-- Higher wins under the 2026-06-20 DESC dedup (oxjob #500).
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date,
    601 as priority
FROM openalex.awards.anrf_awards;

## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_anrf_awards, COUNT(DISTINCT id) as distinct_ids, ROUND(SUM(amount), 2) as total_usd FROM openalex.awards.anrf_awards;

In [ ]:
%sql
SELECT id, display_name, funder_award_id, funder_scheme, funding_type, amount, start_year, end_year,
       lead_investigator.given_name, lead_investigator.family_name, lead_investigator.affiliation.name
FROM openalex.awards.anrf_awards ORDER BY start_year DESC LIMIT 20;

In [ ]:
%sql
SELECT funder_scheme, funding_type, COUNT(*) as cnt, ROUND(SUM(amount), 0) as total_usd
FROM openalex.awards.anrf_awards GROUP BY 1, 2 ORDER BY cnt DESC;

In [ ]:
%sql
SELECT FLOOR(start_year / 10) * 10 AS decade, COUNT(*) as cnt, ROUND(SUM(amount), 0) as usd
FROM openalex.awards.anrf_awards GROUP BY 1 ORDER BY 1;

In [ ]:
%sql
-- Section 6.4a frequency check: PI top-20 must be a real long-tail (Granger has 5 grants 1973-1987 by design).
SELECT lead_investigator.given_name AS given, lead_investigator.family_name AS family, COUNT(*) AS n
FROM openalex.awards.anrf_awards
GROUP BY 1, 2 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
SELECT display_name, COUNT(*) AS n
FROM openalex.awards.anrf_awards
GROUP BY 1 ORDER BY n DESC LIMIT 20;

In [ ]:
%sql
-- Section 6.3 / 6.7 coverage (expect 285 / 100% title, amount, PI, institution, start_year; ~53% description; dates NULL by design).
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    collect_set(currency) as currencies,
    ROUND(MIN(amount), 0) as min_amt,
    ROUND(MAX(amount), 0) as max_amt,
    ROUND(AVG(amount), 0) as avg_amt,
    COUNT(start_year) as has_start_year,
    COUNT(end_year) as has_end_year,
    COUNT(start_date) as has_start_date,
    COUNT(lead_investigator.family_name) as has_pi_name,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    COUNT(landing_page_url) as has_landing_page
FROM openalex.awards.anrf_awards;

In [ ]:
%sql
-- Confirm rows reached the shared raw table at the assigned priority (6.8).
SELECT provenance, priority, COUNT(*) as n
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'anrf_grants_master_list'
GROUP BY provenance, priority;